# Analysis of the provided dataset

------------------------------------

### Initializing the work environment

```bash
python -m venv venv
source venv/bin/activate
pip install -r requirement.txt
```

### Import the necessary libs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

--------------------------------------------

The dataset consists of 3 columns: ID, a secret color, and the testimony. The color will be used to check if our clusters are accurate at the end of the learning phase of our project.

Our objective is to group similar testimonies into clusters, where each cluster potentially corresponds to a specific diagnosis or underlying disease.

In [ ]:
# Load the dataset and give names to the 3 columns (0: ID, 1: Color, 2: Testimony)
def init(path):
    df = pd.read_csv(path, header=None, names=[0, 1, 2])
    # Print the shape to be sure the dataset is loaded perfectly
    print(f"Dataset loaded perfectly : {df.shape}")
    return df

In [ ]:
# function for print the dataset (debug or visualization purpose)
def print_df(n, df):
    df_print = df.head(n)
    print(f"this is the {n} first lines of the dataset:\n\n{df_print}")

-------------------------------

## Dataset analysis

In this notebook, our goal is to understand the data within our dataset as thoroughly as possible. To achieve this, graphical representations and data visualization will be our best tools. This section does not include any machine learning algorithms; it focuses entirely on how our data can be visually interpreted to serve our objective.

### First step : The text average length

We will determine the length of the longest and shortest testimonies, as well as calculate the average word count per patient.

In [ ]:
def mean_testimony(df):
    # We count the number of words in each testimony (skipping the potential header with [1:])
    words_array = df[2][1:].str.split().str.len()
    
    # Get the max, min, and average to understand our data distribution
    max_len = words_array.max()
    min_len = words_array.min()
    mean_len = words_array.mean()

    # Print the stats to check if we have crazy outliers
    print("Testimony Length Analysis : ")
    print(f"    Longest testimony : {max_len} words")
    print(f"    Shortest testimony : {min_len} words")
    print(f"    Average testimony : {mean_len:.2f} words")

# We load the dataset and run the analysis
df = init("./cleaned_dataset.csv")
mean_testimony(df)

We can observe that the variance in text lengths is not too large. Therefore, no further changes or filtering are needed for this part.

### Second step : the vocabulary analysis

We are now going to make a function will show us the most used words, and afterward, we can analyze the results to know what to do with our dataset.

In [ ]:
def most_present_words(df, num=40):
    # Split sentences into lists of words
    words_array = df[2].str.split() 
    # Explode puts every single word on its own row to make counting easier
    all_words_array = words_array.explode() 
    # Count the occurrences and keep only the top 'num' words
    frequence = all_words_array.value_counts().head(num) 
    
    # Setup the visual graph
    plt.figure(figsize=(15, 7))
    plt.bar(frequence.index, frequence.values, color="orange")
    plt.xlabel("Words")
    # Rotate so the words don't overlap on the x-axis
    plt.xticks(rotation=45) 
    plt.ylabel("Number of occurrences")
    plt.show()

most_present_words(df)

We can see that the most written words are useless for the training because the model wants words with a real meaning, like a location or a feeling. We don't need words like 'I', 'my', 'and', etc.

In [ ]:
# We create lists of useless words to remove them and make it easier for the model
# 'ghosts' are leftover letters from the punctuation cleaning
ghosts = ["s", "t", "m", "ve", "d", "ll", "re", "don", "didn", "doesn"]

# 'storytelling' are verbs and words that patients use to tell their story but have no medical value
storytelling = ["feel", "feels", "feeling", "felt", "notice", "noticed", "think", "thought", "believe", "start", "started", "keep", "kept", "got", "get", "went", "go", "came", "come", "make", "made", "like", "just", "gives", "makes", "long", "bit", "night", "turning"]

# We combine standard English stop words with our custom lists
final_stop_words = set(list(ENGLISH_STOP_WORDS) + ghosts + storytelling)

# 1. Our cleaning function applied to a single testimony
def clean_single_text(text):
    if not isinstance(text, str):
        return text
    # Split the text into words
    words = text.lower().split() 
    # Keep the word ONLY if it is not in our useless words list
    cleaned_words = [word for word in words if word not in final_stop_words] 
    # Rebuild the testimony string
    return " ".join(cleaned_words) 

# 2. Apply the cleaning function to the entire testimony column
df[2] = df[2].apply(clean_single_text)

# 3. step to prevent a mysterious line creation 
# If the first line is just the number "2" (a false title leftover), we destroy it!
if str(df[2].iloc[0]).strip() == "2":
    df = df.drop(index=0).reset_index(drop=True)

print_df(10, df)

In [ ]:
# Call the graph function again to see the new top words after cleaning
most_present_words(df)

We can now see that the remaining words make much more sense than before this additional cleaning. We are left with highly specific words used in a strictly medical context. This vocabulary will be extremely useful for our machine learning algorithms.

In [ ]:
# Save the fully cleaned dataset to a new CSV file without the index column
df.to_csv("cleaned_dataset.csv", index=False)